In [1]:
import pandas as pd
import ast
import os
from Bio import SeqIO

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)

genus_name = 'Escherichia'

org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
pla_acc = []
for i in org_data_n.index:
    acc_n = org_data_n['accession'][i]
    pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
    for item in pla_data:
        pla_acc.append(acc_n + '-' + item)
        

acc_bio = pd.read_csv('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/biosample_info.tsv', sep='\t')
acc_time = pd.read_csv('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/assembly_submission_info.tsv', sep='\t')
acc_time["submissionDate"] = pd.to_datetime(acc_time["submissionDate"])

target_dir = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
replicon_data = pd.read_csv(f'{target_dir}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
all_trans = replicon_data[replicon_data['category-pident_90']=='intermediate replicon'].copy()
all_trans['acc_n'] = all_trans['accession'].str.split('-').str[0]
all_acc = set(all_trans['acc_n'])

filted_bio = acc_bio[(acc_bio['accession'].isin(all_acc)) & (acc_bio['srr_list'] != '[]')]
filted_bio = pd.merge(filted_bio, acc_time, how='left', on='accession')
filted_bio = filted_bio.sort_values(by="submissionDate", ascending=False, ignore_index=True)

all_acc = []
for idx in filted_bio.index:
    short_read, long_read = False, False
    srr_list = ast.literal_eval(filted_bio.loc[idx, 'srr_list'])
    for run in srr_list:
        if run['platform'] == 'ILLUMINA':
            short_read = True
        if run['platform'] == 'OXFORD_NANOPORE' or run['platform'] == 'PACBIO_SMRT':
            long_read = True
    if short_read and long_read:
        pass
    else:
        continue

    all_acc.append(filted_bio.loc[idx, 'accession'])

In [2]:
unicycler_data = []
hybracter_data = []
flye_data = []
plasmidness_data = []

for acc_n in all_acc:
    folder = f'/active-data/analysis_results/chr_pla/genus/re-assemble_fraction_records/{genus_name}/{acc_n}'
    try:
        pn_data = pd.read_csv(f'{folder}/unicyle/contig_average_plasmid_fraction.csv')
        plasmidness_data.append(pn_data)
    except:
        pass

    try:
        pn_data = pd.read_csv(f'{folder}/hybracter/contig_average_plasmid_fraction.csv')
        plasmidness_data.append(pn_data)
    except:
        pass
        
    try:
        pn_data = pd.read_csv(f'{folder}/flye/contig_average_plasmid_fraction.csv')
        plasmidness_data.append(pn_data)
    except:
        pass
    
    unicyle_dir = f'/active-data/genome_re-assemble/{acc_n}/assembly_unicycler/assembly.fasta'
    hybracter_dir = f'/active-data/genome_re-assemble/{acc_n}/assembly_hybracter/FINAL_OUTPUT/complete/{acc_n}_final.fasta'
    hybracter_im_dir = f'/active-data/genome_re-assemble/{acc_n}/assembly_hybracter/FINAL_OUTPUT/incomplete/{acc_n}_final.fasta'
    flye_dir = f'/active-data/genome_re-assemble/{acc_n}/assembly_flye/assembly.fasta'
    handle = open(unicyle_dir)
    records = SeqIO.parse(handle, "fasta")
    for seq_record in records:
        temp_uni_info = {'contig': f'{acc_n}-{seq_record.id}'}
        if 'circular=true' in seq_record.description:
            temp_uni_info['topology'] = 'circular'
        else:
            temp_uni_info['topology'] = 'linear'
        unicycler_data.append(pd.DataFrame([temp_uni_info]))
        
    result_exist = False
    for cp in ['complete', 'incomplete']:
        try:
            result = pd.read_csv(f'/active-data/genome_re-assemble/{acc_n}/assembly_hybracter/FINAL_OUTPUT/{cp}/{acc_n}_per_contig_stats.tsv', sep='\t')
            result_exist = True
        except:
            continue
    if not result_exist:
        pass
    elif 'circular' in result.columns:
        for idx, row in result.iterrows():
            if row['circular']:
                temp_hyb_info = {'contig': f'{acc_n}-{row['contig_name']}', 'topology': 'circular'}
            else:
                temp_hyb_info = {'contig': f'{acc_n}-{row['contig_name']}', 'topology': 'linear'}
            hybracter_data.append(pd.DataFrame([temp_hyb_info]))
    else:
        for idx, row in result.iterrows():
            temp_hyb_info = {'contig': f'{acc_n}-{row['contig_name']}', 'topology': 'linear'}
            hybracter_data.append(pd.DataFrame([temp_hyb_info]))

    try:
        result = pd.read_csv(f'/active-data/genome_re-assemble/{acc_n}/assembly_flye/assembly_info.txt', sep='\t')
        for idx, row in result.iterrows():
            if row['circ.'] == 'Y':
                temp_flye_info = {'contig': f'{acc_n}-{row['#seq_name']}', 'topology': 'circular'}
            elif row['circ.'] == 'N':
                temp_flye_info = {'contig': f'{acc_n}-{row['#seq_name']}', 'topology': 'linear'}
            flye_data.append(pd.DataFrame([temp_flye_info]))
    except:
        pass
            
unicycler_data = pd.concat(unicycler_data, ignore_index=True)
unicycler_data['tool'] = 'unicycler'
hybracter_data = pd.concat(hybracter_data, ignore_index=True)
hybracter_data['tool'] = 'hybracter'
flye_data = pd.concat(flye_data, ignore_index=True)
flye_data['tool'] = 'flye'
all_topo = pd.concat([unicycler_data, hybracter_data, flye_data], ignore_index=True)
plasmidness_data = pd.concat(plasmidness_data, ignore_index=True)
plasmidness_data['contig'] = plasmidness_data.apply(lambda x: x['contig'].replace(f"{x['accession']}_", f"{x['accession']}-"), axis=1)

merged_data = pd.merge(plasmidness_data, all_topo, on='contig', how='left')
os.chdir(f'/active-data/analysis_results/chr_pla/genus/re-assemble_fraction_records/{genus_name}')
merged_data.to_csv('all_contig_info.tsv', sep='\t')